In [17]:
import pandas as pd
import re

# 1. CARGA DE DATOS
print("Cargando datos...")
df_acc = pd.read_csv('madrid_ciudad_accidentes.csv', low_memory=False)
df_2024 = pd.read_excel('madrid 2024.xlsx') 
df_sin_trafico = pd.read_csv('madrid_accidentes_final_sinTrafico.csv', low_memory=False, encoding='latin-1')

print("Extrayendo hora...")
#Extraer hora de la fecha en 2024
df_2024['hora'] = df_2024['fecha'].astype(str).str.extract(r'(\d{2}:\d{2}:\d{2})')[0].fillna(df_2024['hora'])
df_2024['fecha'] = df_2024['fecha'].astype(str).str.split(' ').str[0]

# 2. UNIR Y FILTRAR POR CONDUCTORES
print("Filtrando datos...")
df = pd.concat([df_acc, df_2024], ignore_index=True)
df = df[df['tipo_persona'].astype(str).str.contains('Conductor', case=False, na=False)].copy()

# 3. ESTANDARIZAR FORMATO DE HORA A HH:MM:SS
print("Estandarizando formato de hora...")
def format_to_hour_only(h):
    h = str(h).strip().upper()
    # Busca patrones tipo 14:30 o "DE 14:00 A 14:59"
    match = re.search(r'(\d{1,2}):\d{2}', h)
    if match:
        hh = match.group(1).zfill(2)
        return f"{hh}:00:00"

df['hora'] = df['hora'].apply(format_to_hour_only)
dias_map = {0:'LUNES', 1:'MARTES', 2:'MIERCOLES', 3:'JUEVES', 4:'VIERNES', 5:'SABADO', 6:'DOMINGO'}
df['dia_semana'] = pd.to_datetime(df['fecha']).dt.dayofweek.map(dias_map)

# 4. METEOROLOGÍA
print("Procesando datos de tiempo de 2016-2019...")
# Aseguramos que la columna fecha es string para el extract
mask_16_19 = df['fecha'].astype(str).str.extract(r'(\d{4})')[0].fillna('0').astype(int).between(2016, 2019)

# Mapeo meteorológico
def map_weather(row):
    if row.get('CPFA Lluvia') == 'SI': return 'Lluvia débil'
    if row.get('CPFA Nieve') == 'SI': return 'Nevando'
    if row.get('CPFA Granizo') == 'SI': return 'Granizando'
    if row.get('CPFA Niebla') == 'SI': return 'Nublado'
    return 'Despejado'
df.loc[mask_16_19, 'estado_meteorológico'] = df.loc[mask_16_19].apply(map_weather, axis=1)

# 5. CREAR DIRECCIÓN ÚNICA
print("Creando dirección única...")
# Limpieza de caracteres invisibles
df['localizacion'] = df['localizacion'].astype(str).str.replace(r'[\t\r\n]', ' ', regex=True).str.strip()
# Limpieza de número de calle
num = df['numero'].fillna('').astype(str).str.replace('.0', '', regex=False).str.strip()
# Unión final
df['direccion_unica'] = (df['localizacion'] + ' ' + num).str.replace(r'\s+', ' ', regex=True).str.strip()

# 6. ASIGNAR COORDENADAS
print("Asignando coordenadas desde madrid_accidentes_final_sinTrafico.csv...")
for col in ['coordenada_x_utm', 'coordenada_y_utm']:
    # Limpiar cualquier residuo de texto y asegurar formato numérico
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(',', '.').str.strip(), 
        errors='coerce'
    )
for col in ['coordenada_x_utm', 'coordenada_y_utm']:
    # Limpiar cualquier residuo de texto y asegurar formato numérico
    df_sin_trafico[col] = pd.to_numeric(
        df_sin_trafico[col].astype(str).str.replace(',', '.').str.strip(), 
        errors='coerce'
    )

# Preparamos un mapa de coordenadas por número de expediente para rapidez
coords_map_x = df_sin_trafico.drop_duplicates('num_expediente').set_index('num_expediente')['coordenada_x_utm']
coords_map_y = df_sin_trafico.drop_duplicates('num_expediente').set_index('num_expediente')['coordenada_y_utm']

# Identificamos registros de 2016-2019
anio_tmp = df['fecha'].astype(str).str.extract(r'(\d{4})')[0].fillna('0').astype(int)
mask_16_19 = anio_tmp.between(2016, 2019)

# Rellenamos coordenadas solo para los años 2016-2019 donde falten
mask_fill = mask_16_19 & (pd.to_numeric(df['coordenada_x_utm'], errors='coerce').isna() | (df['coordenada_x_utm'] == 0))
df.loc[mask_fill, 'coordenada_x_utm'] = df.loc[mask_fill, 'num_expediente'].map(coords_map_x)
df.loc[mask_fill, 'coordenada_y_utm'] = df.loc[mask_fill, 'num_expediente'].map(coords_map_y)

# 7. UNIFICAR FORMATO DE TEXTO
print("Unificando formato de texto...")
cols_to_lower = ['dia_semana', 'distrito', 'tipo_accidente', 'tipo_vehiculo', 
                 'rango_edad', 'estado_meteorológico', 'sexo']
for col in cols_to_lower:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().str.strip()

# 8. LIMPIEZA DE COLUMNAS
print("Limpiando columnas...")
cols_to_remove = [
    'num_victimas', 'tipo_persona', 'lesividad', 'cod_distrito',
    'cod_lesividad', 'positiva_alcohol', 'positiva_droga', 'localizacion', 'numero'
]
# También eliminamos las columnas meteorológicas antiguas (CPFA/CPSV)
old_cols = [c for c in df.columns if c.startswith('CPFA') or c.startswith('CPSV')]
# Eliminamos 'loc_clean' que es la que creamos nosotros para el cruce
df.drop(columns=cols_to_remove + old_cols, inplace=True, errors='ignore')
print("Guardando archivo limpio...")
df.to_csv('madrid_accidentes_limpio.csv', index=False, encoding='latin-1')

Cargando datos...
Extrayendo hora...
Filtrando datos...
Estandarizando formato de hora...
Procesando datos de tiempo de 2016-2019...
Creando dirección única...
Asignando coordenadas desde madrid_accidentes_final_sinTrafico.csv...
Unificando formato de texto...
Limpiando columnas...
Guardando archivo limpio...
